# 02 · Classical Supervised Learning

Benchmark classification and regression families under one fold-safe preprocessing contract, then analyze imbalance, calibration, errors, latency, and model-selection tradeoffs.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Compare linear, neighbor, margin, tree, ensemble, boosting, and MLP estimators fairly.
- Explain Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, Brier score, and ECE.
- Interpret calibration separately from discrimination.
- Use Pipeline and cross-validation without preprocessing leakage.
- Select a model using more than the largest score.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

In [ ]:
metric_path = METRICS_DIR / "classical.json"
if RUN_TRAINING or not metric_path.exists():
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "scripts/run_classical.py", "--data-dir", str(DATA_DIR), "--profile", "ci", "--output", str(METRICS_DIR / "classical.json")], check=True)
result = json.loads(metric_path.read_text())
print("config", Path("configs/models/classical.yaml").read_text())
print("loaded", metric_path, "dataset", result.get("_run", {}).get("dataset_version"))

In [ ]:
import pandas as pd

classification = pd.DataFrame(result["classification"]).T
regression = pd.DataFrame(result["regression"]).T
display(classification.sort_values("roc_auc", ascending=False))
display(regression.sort_values("r2", ascending=False))

## Metric map

For binary predictions, precision is $TP/(TP+FP)$ and recall is $TP/(TP+FN)$. F1 is their harmonic
mean. ROC-AUC evaluates ranking across thresholds; PR-AUC focuses on the positive class. Brier score
is mean squared probability error. ECE bins probabilities and compares confidence with observed rate.

In [ ]:
label_rate = pd.read_csv(DATA_DIR / "customers.csv.gz").churned.mean()
baseline = {"all_negative_accuracy": 1 - label_rate, "random_pr_auc": label_rate}
baseline

## Accuracy trap

Several models can exceed 95% accuracy by predicting almost no churn. Use the confusion matrix and
recall to see whether the model actually identifies positives.

In [ ]:
view = classification[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]].sort_values("roc_auc")
view.plot.barh(figsize=(10, 7), title="Classification metrics: same split, different conclusions")
plt.xlim(0, 1)
plt.show()

In [ ]:
best = result["best_classifier"]
matrix = classification.loc[best, "confusion_matrix"]
print("best by ROC-AUC", best, "confusion matrix", matrix)
from IPython.display import Image, display
display(Image(filename=result["diagnostic_figure"]))

## Calibration is not ranking

A model may rank churners well but output probabilities that are systematically too high or low.
Lower Brier/ECE is better; compare them only after confirming the score really is probability-like.
The SVC sigmoid margin here is explicitly an educational approximation, not fitted calibration.

In [ ]:
calibration = classification[["roc_auc", "brier", "ece_10_bin"]].sort_values("brier")
display(calibration)
calibration.plot.scatter(x="roc_auc", y="brier", s=70, title="Discrimination vs calibration")
for name, row in calibration.iterrows():
    plt.annotate(name, (row.roc_auc, row.brier), fontsize=8)
plt.show()

## Fold-safe preprocessing

Numeric features use median imputation and scaling; categorical features use most-frequent
imputation and one-hot encoding. Because these transformations live inside Pipeline, every CV fold
fits its own statistics using training rows only.

In [ ]:
from rookie_lab.classical.preprocessing import preprocessor

print(preprocessor())
print("CV folds", result["cross_validation"]["folds"])
print("CV mean", result["cross_validation"]["logistic_roc_auc_mean"])

## Imbalanced-learn evidence

Random oversampling changes the training distribution to 50% positive. It must happen only inside
training folds in a real tuning workflow. Oversampling before the split duplicates information into
validation and produces optimistic results.

In [ ]:
pd.Series(result["class_imbalance"]).plot.bar(title="Positive rate before/after resampling")
plt.ylim(0, 0.55)
plt.show()

## Regression task

MAE is in currency units and robust to large residuals; RMSE penalizes large misses more strongly;
$R^2$ compares residual variance with a mean-target baseline and can be negative.

In [ ]:
display(regression[["mae", "rmse", "r2", "fit_seconds", "inference_ms_per_1000"]].sort_values("r2", ascending=False))

## Runtime and deployment tradeoff

Training speed, batch latency, calibration, interpretability, and dependency size matter. The best
ROC-AUC model is not automatically the correct product model.

In [ ]:
efficiency = classification.explode("confusion_matrix") if False else classification
efficiency.plot.scatter(x="inference_ms_per_1000", y="roc_auc", s=80, logx=True)
for name, row in efficiency.iterrows():
    plt.annotate(name, (row.inference_ms_per_1000, row.roc_auc), fontsize=8)
plt.title("ROC-AUC versus inference time")
plt.show()

## Error-slice exercise

Load `artifacts/models/best_classical.joblib`, score the held-out set, and compare recall by region,
channel, and tenure quartile. A global metric can hide a failing subgroup. This repository does not
claim fairness from synthetic demographic slices.

In [ ]:
print("Leaderboard")
display(pd.DataFrame(result["classification_leaderboard"]))
print("Local XGBoost backend:", result["xgboost_backend"])

## Exercises and interview checkpoint

1. Tune the decision threshold for recall ≥ 0.70 and report precision.
2. Wrap SVC in `CalibratedClassifierCV` and compare Brier/ECE.
3. Explain why Lasso and Ridge are almost tied on the synthetic spend target.
4. Replace random CV with an out-of-time split.
5. Choose one model for batch scoring and defend the choice without saying “highest AUC.”